In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

In [5]:
df = pd.read_csv("level_cleaned.csv")
df.head()

,State/UT,Primary Drop Out Rate - Boys,Primary Drop Out Rate - Girls,Primary Drop Out Rate - Overall,Upper Primary Drop Out Rate - Boys,Upper Primary Drop Out Rate - Girls,Upper Primary Drop Out Rate - Overall,Secondary Drop Out Rate - Boys,Secondary Drop Out Rate - Girls,Secondary Drop Out Rate - Overall
0,Andaman And Nicobar Islands,0.2,0.7,0.4,0.9,1.0,1.0,6.0,3.9,5.0
1,Andhra Pradesh,0.0,0.0,0.0,1.7,1.5,1.6,17.5,15.0,16.3
2,Arunachal Pradesh,9.3,9.2,9.3,4.8,8.4,6.7,11.2,12.3,11.7
3,Assam,6.8,5.2,6.0,10.1,7.6,8.8,19.8,20.7,20.3
4,Bihar,0.0,0.0,0.0,4.0,5.2,4.6,19.5,21.4,20.5


In [7]:
df.columns = df.columns.str.strip()

# Features (X)
X = df[
    [
        "Primary Drop Out Rate - Boys",
        "Primary Drop Out Rate - Girls",
        "Primary Drop Out Rate - Overall",
        "Upper Primary Drop Out Rate - Boys",
        "Upper Primary Drop Out Rate - Girls",
        "Upper Primary Drop Out Rate - Overall"
    ]
]
# Target (y)
y = df["Secondary Drop Out Rate - Overall"]

In [8]:
### 5-Fold Cross Validation
# To obtain a more reliable estimate of model performance, 5 fold cross validation was performed on the Linear Regression model. The model was trained and evaluated five times using different subsets of the dataset
cv_scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5,
    scoring="r2"
)
print("Cross Validation R² Scores:")
print(cv_scores)
print("\nAverage R² Score:")
print(cv_scores.mean())
print("\nStandard Deviation:")
print(cv_scores.std())
# The cross-validation results showed considerable variation in R² scores across the folds. While some folds achieved moderate predictive performance, others produced negative R² values, indicating poor generalization on those subsets. The average R² score was -0.204 with a standard deviation of 0.605.
# These findings suggest that the Linear Regression model is sensitive to the composition of the training and testing data. The variability is likely influenced by the small dataset size (36 observations) and the complexity of the relationships among the educational indicators

Cross Validation R² Scores:
[ 0.42239462  0.36660653 -0.20759833 -0.35282241 -1.24793701]

Average R² Score:
-0.20387131961960905

Standard Deviation:
0.6049673537024578


In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid = { "max_depth": [2, 3, 4, 5, None],
    "min_samples_split": [2, 4, 6],
        "min_samples_leaf": [1, 2, 3] }

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)
grid_search.fit(X, y)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, None],
                         'min_samples_leaf': [1, 2, 3],
                         'min_samples_split': [2, 4, 6]},
             scoring='r2')

In [11]:
print("Best Parameters:")
print(grid_search.best_params_)
print("\nBest Cross-Validation R² Score:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}

Best Cross-Validation R² Score:
-0.24772306580555198


In [13]:
best_tree = grid_search.best_estimator_
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
best_tree.fit(X_train, y_train)
y_pred = best_tree.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


MAE: 6.547083333333333
RMSE: 8.672417683219727
R²: -1.522372045685327


In [ ]:
## Hyperparameter Tuning using Grid Search
# Grid Search with 5 fold cross-validation was applied to identify the optimal hyperparameters for the Decision Tree Regressor. The search considered different values of maximum tree depth (`max_depth`), minimum samples required to split a node (`min_samples_split`), and minimum samples required at a leaf node (`min_samples_leaf`).
# The best performing parameter combination was:
#- max_depth = 5
#- min_samples_split = 2
#- min_samples_leaf = 1
# The tuned model achieved a cross validation R² score of -0.248. When evaluated on the test dataset, the tuned Decision Tree produced an MAE of 6.547, an RMSE of 8.672, and an R² score of -1.522.
# Compared with the default Decision Tree model developed in Week 5, the tuned model did not improve predictive performance. This suggests that the dataset is too small for the Decision Tree to benefit significantly from hyperparameter tuning and that the selected features may not adequately explain the target variable.

In [14]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2, 3]
}
random_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)
random_search.fit(X, y)

RandomizedSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'max_depth': [2, 3, 4, 5, None],
                                        'min_samples_leaf': [1, 2, 3],
                                        'min_samples_split': [2, 4, 6]},
                   random_state=42, scoring='r2')

In [15]:
print("Best Parameters:")
print(random_search.best_params_)
print("\nBest Cross-Validation R² Score:")
print(random_search.best_score_)

Best Parameters:
{'min_samples_split': 4, 'min_samples_leaf': 1, 'max_depth': 4}

Best Cross-Validation R² Score:
-0.42782589918675107


In [16]:
best_random_tree = random_search.best_estimator_
best_random_tree.fit(X_train, y_train)
y_pred_random = best_random_tree.predict(X_test)
mae_random = mean_absolute_error(y_test, y_pred_random)
rmse_random = np.sqrt(mean_squared_error(y_test, y_pred_random))
r2_random = r2_score(y_test, y_pred_random)
print("MAE:", mae_random)
print("RMSE:", rmse_random)
print("R²:", r2_random)

MAE: 6.54
RMSE: 9.030101051483312
R²: -1.734727089796261


In [ ]:
## Hyperparameter Tuning using Random Search
# Random Search was applied as an alternative hyperparameter optimization technique for the Decision Tree Regressor. Unlike Grid Search, which evaluates every possible parameter combination, Random Search evaluates a randomly selected subset of parameter combinations.
# The best parameter combination identified by Random Search was:
# - max_depth = 4
# - min_samples_split = 4
# - min_samples_leaf = 1
# The tuned model achieved a cross validation R² score of -0.428. When evaluated on the test dataset, the model obtained an MAE of 6.540, an RMSE of 9.030, and an R² score of -1.735.
# Comparison with the default Decision Tree and the Grid Search tuned model showed that Random Search did not improve predictive performance. The default Decision Tree remained the best performing Decision Tree model for this dataset.

## Evaluation Metrics

The objective of this project is to predict dropout rates, which are continuous numerical values. Therefore, this is a regression problem rather than a classification problem.

For regression models, the following evaluation metrics were used:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between the actual and predicted values. Lower values indicate better predictive accuracy.

- **Root Mean Squared Error (RMSE):** Measures the square root of the average squared prediction errors. RMSE penalizes larger errors more heavily than MAE.

- **Coefficient of Determination (R²):** Indicates the proportion of variance in the target variable explained by the model. Higher R² values represent better model performance, while negative values indicate that the model performs worse than predicting the mean target value.

Accuracy, Precision, Recall, and F1 score are designed for classification problems and were therefore not used in this regression based study.

## Week 6 Conclusion

During Week 6, model performance was further evaluated using 5 fold cross validation and hyperparameter tuning techniques.

Cross validation revealed that the Linear Regression model exhibited considerable variability across different folds, with an average R² score of -0.204. This indicates limited generalization due to the small dataset size.

Hyperparameter tuning of the Decision Tree Regressor was performed using both Grid Search and Random Search. Although both methods identified optimal parameter combinations, neither approach improved the model's predictive performance compared to the default Decision Tree. The default Decision Tree achieved lower prediction errors and a higher R² score than the tuned models.

Overall, these results suggest that the primary limitation lies in the size and characteristics of the dataset rather than in the choice of hyperparameters. The experiments also demonstrate the importance of model validation and hyperparameter optimization in assessing machine learning models.